Korzystanie z narzędzi generatywnej AI w rozwiązywaniu zadań nie jest dozwolone

<img src="no_AI.png" alt="Use of AI allowed only when properly documented " width="100" height="100">

# Zadanie obowiązkowe [0-10] pkt

1. [0-0.5 pkt] Użyj zbalansowania klas w regresji logistycznej (parametr `class_weight`). Porównaj z modelem podstawowym
1. [0-1 pkt] Porównaj różne techniki podpróbkownia (*undersampling*). W szczególności (weź pod uwagę hiperparametry każdego podejścia!):
   1. Podpróbkowanie losowe
   2. [Cluster Centroids](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.ClusterCentroids.html#clustercentroids)
   3. *Edited Nearest Neighbours* (patrz ćwiczenia)
1. [0-2 pkt] Porównaj różne techniki nadpróbkowania (*oversampling*). W szczególności (weź pod uwagę hiperparametry każdego podejścia!):
   1. Nadpróbkowanie losowe
   2. [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html)
   3. [Borderline SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.BorderlineSMOTE.html#)
   3. [ADASYN](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.ADASYN.html)
1. [0-1 pkt] Przeprowadź dostrajanie progu (*threshold tuning*) tj. użyj innych progów odcięcia klasy pozytywnej w regresji logistycznej. Możesz wykorzystać np. metody opisane [tutaj](https://scikit-learn.org/stable/modules/classification_threshold.html). Optymalizuj względem AUPRC
1. [0-1 pkt] Użyj przynajmniej jednego innego klasyfikatora poznanego na zajęciach i porównaj jego wyniki z regresją logistyczną
1. [0-1.5 pkt] Zaimplementuj fokalną funkcję kosztu (*focal loss*) w `scikit-learn`. W tym celu, wyjątkowo, **możesz skorzystać z narzędzi generatywnej sztucznej inteligencji**
1. [0-2 pkt] Sporządź analizę porównawczą technik / modeli wyżej, uwzględniając podejścia podstawowe (*baseline*). Zwizualizuj krzywe ROC i PRC
1. [0-1 pkt] Skomentuj uzyskane wyniki

In [1]:
!pip install ucimlrepo

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.under_sampling import EditedNearestNeighbours
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [3]:
sklearn.set_config(transform_output="pandas")
sklearn.__version__

'1.6.1'

### Załadowanie danych

In [4]:
bank_marketing = fetch_ucirepo(id=222)
X = bank_marketing.data.features
y = bank_marketing.data.targets['y']

In [5]:
X_sel = X[['balance', 'duration', 'education']]
idx = X_sel.dropna().index
X_sel = X_sel.loc[idx]
y_sel = y.loc[idx]

In [6]:
label_encoder = LabelEncoder()
y_trans = label_encoder.fit_transform(y_sel.values)

In [7]:
enc = OneHotEncoder(drop='first', sparse_output=False)
X_trans = X_sel.drop(columns=['education']).join(
    enc.fit_transform(X_sel[['education']]))

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_trans, y_trans,
                                                    test_size=0.2, random_state=0)

### Regresja logistyczna


In [9]:
logreg = LogisticRegression(random_state=0).fit(X_train, y_train)
y_train_pred = logreg.predict(X_train)
y_train_pred_proba = logreg.predict_proba(X_train)[:, 1]
y_test_pred = logreg.predict(X_test)
y_test_pred_proba = logreg.predict_proba(X_test)[:, 1]

In [10]:
confusion_matrix(y_train, y_train_pred)

array([[30173,   452],
       [ 3358,   700]])

In [11]:
confusion_matrix(y_test, y_test_pred)

array([[7570,  122],
       [ 825,  154]])

In [12]:
roc_auc_score(y_train, y_train_pred_proba), roc_auc_score(y_test, y_test_pred_proba)

(np.float64(0.8148638657828828), np.float64(0.8148813592993158))

In [13]:
average_precision_score(y_train, y_train_pred_proba), average_precision_score(y_test, y_test_pred_proba)

(np.float64(0.40083384565216906), np.float64(0.37618735285442156))

### Zadanie 1

In [14]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0,1])
weight = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
print(f"Waga klasy 0: {weight[0]:.4f}")
print(f"Waga klasy 1: {weight[1]:.4f}")

Waga klasy 0: 0.5663
Waga klasy 1: 4.2734


In [15]:
logreg_bal = LogisticRegression(random_state=0, class_weight='balanced').fit(X_train, y_train)

In [24]:
bal_y_train_pred = logreg_bal.predict(X_train)
bal_y_train_proba = logreg_bal.predict_proba(X_train)[:, 1]
bal_y_test_pred = logreg_bal.predict(X_test)
bal_y_test_proba = logreg_bal.predict_proba(X_test)[:, 1]

In [25]:
confusion_matrix(y_train, bal_y_train_pred)

array([[24876,  5749],
       [ 1440,  2618]])

In [26]:
confusion_matrix(y_test, bal_y_test_pred)

array([[6245, 1447],
       [ 343,  636]])

In [29]:
roc_auc_score(y_train, bal_y_train_proba), roc_auc_score(y_test, bal_y_test_proba)

(np.float64(0.8161699801852726), np.float64(0.8170468289620246))

In [30]:
average_precision_score(y_train, bal_y_train_proba), average_precision_score(y_test,bal_y_test_proba)

(np.float64(0.4008202225715665), np.float64(0.37638942603907105))

Wyniki auroc i auprc są prawie identyczne z baseline, natomiast zmianę widać w macierzach pomyłek. Widać, że model po zbalansowaniu znacznie częściej daje wynik positive. Dużo częściej (prawie 4x) łapie true positive, ale także pojawiło się znacznie więcej false positive.

### Zadanie 2

#### Próbkowanie losowe

In [34]:
from imblearn.under_sampling import RandomUnderSampler

sampling_ratios = [1.0,0.5,0.3,0.2]

for ratio in sampling_ratios:
  rus = RandomUnderSampler(sampling_strategy= ratio,random_state=0)
  X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
  print("Ratio: ",ratio)
  print(np.unique(y_train_rus, return_counts=True))
  logreg_rus = LogisticRegression(random_state=0).fit(X_train_rus, y_train_rus)
  rus_y_train_pred = logreg_rus.predict(X_train_rus)
  rus_y_train_proba = logreg_rus.predict_proba(X_train_rus)[:, 1]
  rus_y_test_pred = logreg_rus.predict(X_test)
  rus_y_test_proba = logreg_rus.predict_proba(X_test)[:, 1]
  print("Train")
  print(confusion_matrix(y_train_rus, rus_y_train_pred))
  print("Test")
  print(confusion_matrix(y_test, rus_y_test_pred))
  print("auroc")
  print(roc_auc_score(y_train_rus, rus_y_train_proba),roc_auc_score(y_test, rus_y_test_proba))
  print("auprc")
  print(average_precision_score(y_train_rus, rus_y_train_proba),average_precision_score(y_test, rus_y_test_proba))

Ratio:  1.0
(array([0, 1]), array([4058, 4058]))
Train
[[3303  755]
 [1437 2621]]
Test
[[6224 1468]
 [ 345  634]]
auroc
0.8144938072662996 0.816819485854
auprc
0.8018746803521457 0.373557042826695
Ratio:  0.5
(array([0, 1]), array([8116, 4058]))
Train
[[7434  682]
 [2262 1796]]
Test
[[7057  635]
 [ 548  431]]
auroc
0.8129846404075358 0.8152290136549283
auprc
0.6798715101573235 0.37333806803481534
Ratio:  0.3
(array([0, 1]), array([13526,  4058]))
Train
[[12931   595]
 [ 2692  1366]]
Test
[[7362  330]
 [ 674  305]]
auroc
0.8154354733052682 0.8149946324717137
auprc
0.5842028446074035 0.3754363293525816
Ratio:  0.2
(array([0, 1]), array([20290,  4058]))
Train
[[19764   526]
 [ 3051  1007]]
Test
[[7487  205]
 [ 763  216]]
auroc
0.8153126207691772 0.8148720637283101
auprc
0.4959232715216065 0.376516378556372


Auroc jest prawie identyczny dla wszystkich, losowe podpróbkowanie nie poprawia jego wyników. Auprc jest podobny do baseline na zbiorach testowych, ale na zbiorach treningowych widać różnicę. Przy większej liczbie próbek, na zbiorach trenigowych są uzyskiwane lepsze wyniki, natomiast nie przekłada się to na zbiór testowy. Oznacza to że model dopasowuje się do danych ale ma problemy z generalizacją.

#### Cluster Centroids

In [35]:
from imblearn.under_sampling import ClusterCentroids

sampling_ratios = [1.0,0.5,0.3,0.2]

for ratio in sampling_ratios:
  cc = ClusterCentroids(sampling_strategy= ratio,random_state=0)
  X_train_cc, y_train_cc = cc.fit_resample(X_train, y_train)
  print("Ratio: ",ratio)
  print(np.unique(y_train_cc, return_counts=True))
  logreg_cc = LogisticRegression(random_state=0).fit(X_train_cc, y_train_cc)
  cc_y_train_pred = logreg_cc.predict(X_train_cc)
  cc_y_train_proba = logreg_cc.predict_proba(X_train_cc)[:, 1]
  cc_y_test_pred = logreg_cc.predict(X_test)
  cc_y_test_proba = logreg_cc.predict_proba(X_test)[:, 1]
  print("Train")
  print(confusion_matrix(y_train_cc, cc_y_train_pred))
  print("Test")
  print(confusion_matrix(y_test, cc_y_test_pred))
  print("auroc")
  print(roc_auc_score(y_train_cc, cc_y_train_proba),roc_auc_score(y_test, cc_y_test_proba))
  print("auprc")
  print(average_precision_score(y_train_cc, cc_y_train_proba),average_precision_score(y_test, cc_y_test_proba))

Ratio:  1.0
(array([0, 1]), array([4058, 4058]))
Train
[[2868 1190]
 [1421 2637]]
Test
[[5200 2492]
 [ 373  606]]
auroc
0.7164137502517101 0.7147288853760484
auprc
0.6749655451409813 0.2995578311178581
Ratio:  0.5
(array([0, 1]), array([8116, 4058]))
Train
[[7451  665]
 [2947 1111]]
Test
[[7412  280]
 [ 723  256]]
auroc
0.7317818595617368 0.7532457478074405
auprc
0.5486989696503914 0.3341228551795723
Ratio:  0.3
(array([0, 1]), array([13526,  4058]))
Train
[[13040   486]
 [ 3295   763]]
Test
[[7549  143]
 [ 805  174]]
auroc
0.7522913266288819 0.7862543204486094
auprc
0.46906757948869604 0.36028890603608144
Ratio:  0.2
(array([0, 1]), array([20290,  4058]))
Train
[[19840   450]
 [ 3360   698]]
Test
[[7565  127]
 [ 819  160]]
auroc
0.7813357195480709 0.8068012505995643
auprc
0.42901862223716913 0.37274558389753615


Podpróbkowanie Cluster Centroids daje gorsze wyniki niż RUS, tym gorsze im więcej próbek jest usuwane (wyższe ratio). Im niższe ratio - tym samym zbiór treningowy jest bardziej podobny do baseline - tym wyższe wyniki, bliższe do tych z baseline.

#### Edited Nearest Neighbours

In [38]:
neighbours_count = [3,5,7]

for neighbors in neighbours_count:
  enn = EditedNearestNeighbours(n_neighbors=neighbors)
  X_train_enn, y_train_enn = enn.fit_resample(X_train, y_train)
  logreg_enn = LogisticRegression(random_state=0,max_iter=1000).fit(X_train_enn, y_train_enn)
  enn_y_train_pred = logreg_enn.predict(X_train_enn)
  enn_y_train_proba = logreg_enn.predict_proba(X_train_enn)[:, 1]
  enn_y_test_pred = logreg_enn.predict(X_test)
  enn_y_test_proba = logreg_enn.predict_proba(X_test)[:, 1]
  print(np.unique(y_train_enn, return_counts=True))
  print("Train")
  print(confusion_matrix(y_train_enn, enn_y_train_pred))
  print("Test")
  print(confusion_matrix(y_test, enn_y_test_pred))
  print("auroc")
  print(roc_auc_score(y_train_enn, enn_y_train_proba),roc_auc_score(y_test, enn_y_test_proba))
  print("auprc")
  print(average_precision_score(y_train_enn, enn_y_train_proba),average_precision_score(y_test, enn_y_test_proba))

(array([0, 1]), array([23708,  4058]))
Train
[[23253   455]
 [ 2362  1696]]
Test
[[7153  539]
 [ 573  406]]
auroc
0.8723376643112194 0.8192095099534319
auprc
0.6500107379496832 0.37828399335267227
(array([0, 1]), array([20894,  4058]))
Train
[[20404   490]
 [ 2082  1976]]
Test
[[6899  793]
 [ 498  481]]
auroc
0.8915292546861548 0.8193235798890587
auprc
0.7236727525094092 0.37834792973741543
(array([0, 1]), array([18806,  4058]))
Train
[[18323   483]
 [ 1867  2191]]
Test
[[6698  994]
 [ 452  527]]
auroc
0.9049384989124252 0.8196762804117884
auprc
0.7692225704051746 0.37830863822831295


ENN usuwa najmniej próbek ze wszystkich podejść, wyniki są natomiast bardzo podobne jak dla baseline. Poprawił się wynik auroc dla zbioru treningowego, ale bez wpływu na wyniki na zbiorze testowym.

### Zadanie 3

#### Nadpróbkowanie losowe

In [39]:
from imblearn.over_sampling import RandomOverSampler

sampling_ratios = [1.0,0.5,0.3]

for ratio in sampling_ratios:
  ros = RandomOverSampler(sampling_strategy= ratio,random_state=0)
  X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)
  print("Ratio: ",ratio)
  print(np.unique(y_train_ros, return_counts=True))
  logreg_ros = LogisticRegression(random_state=0).fit(X_train_ros, y_train_ros)
  ros_y_train_pred = logreg_ros.predict(X_train_ros)
  ros_y_train_proba = logreg_ros.predict_proba(X_train_ros)[:,1]
  ros_y_test_pred = logreg_ros.predict(X_test)
  ros_y_test_proba = logreg_ros.predict_proba(X_test)[:,1]
  print("Train")
  print(confusion_matrix(y_train_ros, ros_y_train_pred))
  print("Test")
  print(confusion_matrix(y_test, ros_y_test_pred))
  print("auroc")
  print(roc_auc_score(y_train_ros, ros_y_train_proba),roc_auc_score(y_test, ros_y_test_proba))
  print("auprc")
  print(average_precision_score(y_train_ros, ros_y_train_proba), average_precision_score(y_test, ros_y_test_proba))

Ratio:  1.0
(array([0, 1]), array([30625, 30625]))
Train
[[24865  5760]
 [10868 19757]]
Test
[[6241 1451]
 [ 343  636]]
auroc
0.8156458446314034 0.8170974234270698
auprc
0.804868734757664 0.3766725980337818
Ratio:  0.5
(array([0, 1]), array([30625, 15312]))
Train
[[28169  2456]
 [ 8390  6922]]
Test
[[7035  657]
 [ 541  438]]
auroc
0.8162804800290022 0.8163621437605206
auprc
0.6878616214507441 0.37658520088998815
Ratio:  0.3
(array([0, 1]), array([30625,  9187]))
Train
[[29324  1301]
 [ 6077  3110]]
Test
[[7371  321]
 [ 678  301]]
auroc
0.8139864520184911 0.814809252227086
auprc
0.5799125256626176 0.3760448936618489


Wyniki są prawie takie same jak baseline, znowu widoczna znaczna różnica między test i train w auprc, przy ratio 1.0 - model jest przeuczony.

### SMOTE

In [40]:
from imblearn.over_sampling import SMOTE

params = [(1.0, 3), (1.0, 5), (1.0, 7), (0.5, 5)]

for ratio, neighbors in params:
  smote = SMOTE(sampling_strategy=ratio, k_neighbors=neighbors, random_state=0)
  X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
  print("Ratio: ",ratio)
  print("Neighbors: ",neighbors)
  print(np.unique(y_train_smote, return_counts=True))
  logreg_smote = LogisticRegression(random_state=0, max_iter=1000).fit(X_train_smote, y_train_smote)
  smote_y_train_pred = logreg_smote.predict(X_train_smote)
  smote_y_train_proba = logreg_smote.predict_proba(X_train_smote)[:,1]
  smote_y_test_pred = logreg_smote.predict(X_test)
  smote_y_test_proba = logreg_smote.predict_proba(X_test)[:,1]
  print("Train")
  print(confusion_matrix(y_train_smote, smote_y_train_pred))
  print("Test")
  print(confusion_matrix(y_test, smote_y_test_pred))
  print("auroc")
  print(roc_auc_score(y_train_smote, smote_y_train_proba),roc_auc_score(y_test, smote_y_test_proba))
  print("auprc")
  print(average_precision_score(y_train_smote, smote_y_train_proba), average_precision_score(y_test, smote_y_test_proba))

Ratio:  1.0
Neighbors:  3
(array([0, 1]), array([30625, 30625]))
Train
[[24808  5817]
 [10781 19844]]
Test
[[6207 1485]
 [ 340  639]]
auroc
0.8168980188921282 0.8146350266676653
auprc
0.804382929897697 0.3756541885501372
Ratio:  1.0
Neighbors:  5
(array([0, 1]), array([30625, 30625]))
Train
[[24822  5803]
 [10684 19941]]
Test
[[6204 1488]
 [ 340  639]]
auroc
0.8191311881382757 0.8147047434502078
auprc
0.805806104261607 0.3752944762493198
Ratio:  1.0
Neighbors:  7
(array([0, 1]), array([30625, 30625]))
Train
[[24799  5826]
 [10808 19817]]
Test
[[6208 1484]
 [ 340  639]]
auroc
0.8181424635735111 0.8154185105095726
auprc
0.8047719694515709 0.375441579201061
Ratio:  0.5
Neighbors:  5
(array([0, 1]), array([30625, 15312]))
Train
[[28150  2475]
 [ 8411  6901]]
Test
[[7032  660]
 [ 543  436]]
auroc
0.8186921128953148 0.8149587781264059
auprc
0.688166763959793 0.3761399724945248


SMOTE daje podobne wyniki co ROS. SMOTE jest niedostosowane do zmiennych binarnych, robi interpolacje między próbkami, co dla zmiennych kategorycznych daje nic nie wnoszące wyniki.